In [42]:
import time
import pandas as pd
import numpy as np
import joblib
import optuna
import warnings

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import root_mean_squared_error

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

In [43]:
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [44]:
print("Loading the data")
train = pd.read_csv("/Users/tanmayagarwal/Documents Local/SIH/ir_train.csv")


Loading the data


In [45]:
import Feature_Engineering as en

In [46]:
train = en.engineer(train, is_training=True)

train['primary_delay_cause'] = train['primary_delay_cause'].fillna("Normal_Running")

TARGET = 'delay_minutes'
DROP = ['journey_id', 'departure_date', 'is_delayed', 'delay_minutes']

In [47]:
cat_col = [c for c in train.select_dtypes(include=['object', 'string', 'category']).columns if c not in DROP]
le_map = {}

for col in cat_col:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str)).astype(int)
    le_map[col] = le

joblib.dump(le_map, 'label_encoders.pkl')


['label_encoders.pkl']

In [48]:
FEATURES = [c for c in train.columns if c not in DROP]

X = train[FEATURES].fillna(-999)
y = train[TARGET].astype(float)
cat_idxs = [FEATURES.index(c) for c in cat_col if c in FEATURES]



In [49]:
import time
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

In [50]:
import time
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

def run_oof_regression(model_fn, model_label, X, y, n_folds=5, seed=42):
    oof = np.zeros(len(X))
    rmses = []
    t0 = time.time()
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        vp = model_fn(X_tr, y_tr, X_val, y_val, seed, fold)
        oof[val_idx] = vp
        rmses.append(root_mean_squared_error(y_val, vp))

    # Calculate final metrics across all predictions
    final_rmse = root_mean_squared_error(y, oof)
    final_mae = mean_absolute_error(y, oof)
    final_r2 = r2_score(y, oof)
    
    # Custom "ETA Accuracy": What % of predictions are within 15 minutes of reality?
    tolerance_minutes = 15
    within_window = np.abs(y - oof) <= tolerance_minutes
    eta_accuracy = np.mean(within_window) * 100
    
    mins = (time.time() - t0) / 60
    
    print(f'✅ {model_label:12s} | RMSE: {final_rmse:.1f}m | MAE: {final_mae:.1f}m | R²: {final_r2:.3f} | Accuracy (±15m): {eta_accuracy:.1f}% | {mins:.1f} min')
    return oof

In [51]:
def lgb_fn(X_tr, y_tr, X_val, y_val, seed, fold):
    model = lgb.LGBMRegressor(
        objective='regression', metric='rmse', n_estimators=600,
        learning_rate=0.05, num_leaves=63, random_state=seed, n_jobs=-1, verbose=-1
    )
    model.fit(X_tr, y_tr)
    return model.predict(X_val)

def xgb_fn(X_tr, y_tr, X_val, y_val, seed, fold):
    model = xgb.XGBRegressor(
        objective='reg:squarederror', eval_metric='rmse', n_estimators=500,
        learning_rate=0.05, max_depth=6, random_state=seed, n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    return model.predict(X_val)

def cat_fn(X_tr, y_tr, X_val, y_val, seed, fold):
    model = CatBoostRegressor(
        loss_function='RMSE', eval_metric='RMSE', iterations=600,
        learning_rate=0.05, depth=6, random_seed=seed, verbose=0
    )
    # CatBoost natively handles categorical indices
    model.fit(X_tr, y_tr, cat_features=cat_idxs, verbose=False)
    return model.predict(X_val)

In [52]:
print("Training the Model")
oof_lgb = run_oof_regression(lgb_fn, 'LightGBM', X, y)
oof_xgb = run_oof_regression(xgb_fn, 'XGBoost', X, y)
oof_cat = run_oof_regression(cat_fn, 'CatBoost', X, y)

Training the Model
✅ LightGBM     | RMSE: 29.9m | MAE: 19.6m | R²: 0.796 | Accuracy (±15m): 51.5% | 1.5 min
✅ XGBoost      | RMSE: 29.9m | MAE: 19.6m | R²: 0.796 | Accuracy (±15m): 51.5% | 1.5 min
✅ CatBoost     | RMSE: 29.8m | MAE: 19.6m | R²: 0.797 | Accuracy (±15m): 51.4% | 18.5 min


In [53]:
print("\n🧬 Optimizing Ensemble Weights via Optuna...")

def objective(trial):
    # Suggest weights between 0 and 1
    w_lgb = trial.suggest_float('w_lgb', 0, 1)
    w_xgb = trial.suggest_float('w_xgb', 0, 1)
    w_cat = trial.suggest_float('w_cat', 0, 1)
    
    # Normalize weights so they sum to 1.0
    total = w_lgb + w_xgb + w_cat
    if total == 0: total = 1e-5
    w_lgb /= total
    w_xgb /= total
    w_cat /= total
    
    # Blend predictions
    blended_oof = (w_lgb * oof_lgb) + (w_xgb * oof_xgb) + (w_cat * oof_cat)
    return root_mean_squared_error(y, blended_oof)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=100) # 100 trials is fast and highly effective

# Extract optimal normalized weights
best_w = study.best_params
tot = sum(best_w.values())
weights = {k: v/tot for k, v in best_w.items()}

# --- NEW: Calculate final accuracy metrics for the Blended Meta-Model ---
best_blended_oof = (weights['w_lgb'] * oof_lgb) + \
                   (weights['w_xgb'] * oof_xgb) + \
                   (weights['w_cat'] * oof_cat)

final_rmse = root_mean_squared_error(y, best_blended_oof)
final_mae = mean_absolute_error(y, best_blended_oof)
final_r2 = r2_score(y, best_blended_oof)

# Custom "ETA Accuracy": within ± 15 minutes
tolerance_minutes = 15
within_window = np.abs(y - best_blended_oof) <= tolerance_minutes
eta_accuracy = np.mean(within_window) * 100

print(f"\n🏆 META-MODEL OPTIMIZATION COMPLETE")
print(f"📊 Optimal Weights -> LGBM: {weights['w_lgb']:.3f} | XGB: {weights['w_xgb']:.3f} | CAT: {weights['w_cat']:.3f}")
print(f"🎯 Final Ensemble  | RMSE: {final_rmse:.1f}m | MAE: {final_mae:.1f}m | R²: {final_r2:.3f} | Accuracy (±15m): {eta_accuracy:.1f}%")


🧬 Optimizing Ensemble Weights via Optuna...

🏆 META-MODEL OPTIMIZATION COMPLETE
📊 Optimal Weights -> LGBM: 0.092 | XGB: 0.050 | CAT: 0.858
🎯 Final Ensemble  | RMSE: 29.8m | MAE: 19.6m | R²: 0.797 | Accuracy (±15m): 51.4%


In [54]:
best_w = study.best_params
tot = sum(best_w.values())
weights = {k: v/tot for k, v in best_w.items()}

print(f"✅ Best Blend RMSE: {study.best_value:.2f} mins")
print(f"📊 Optimal Weights -> LGBM: {weights['w_lgb']:.3f} | XGB: {weights['w_xgb']:.3f} | CAT: {weights['w_cat']:.3f}")



✅ Best Blend RMSE: 29.84 mins
📊 Optimal Weights -> LGBM: 0.092 | XGB: 0.050 | CAT: 0.858


In [55]:
print("\n💾 Training Final Production Models on 100% Data...")

final_lgb = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=63, verbose=-1).fit(X, y)
final_xgb = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6).fit(X, y)
final_cat = CatBoostRegressor(iterations=600, learning_rate=0.05, depth=6, verbose=0).fit(X, y, cat_features=cat_idxs)




💾 Training Final Production Models on 100% Data...


In [56]:
joblib.dump(final_lgb, 'lgb_base.pkl')
joblib.dump(final_xgb, 'xgb_base.pkl')
joblib.dump(final_cat, 'cat_base.pkl')
joblib.dump(weights, 'optuna_weights.pkl') # Save the weights instead of a Ridge model
joblib.dump(FEATURES, 'features_list.pkl')


['features_list.pkl']